# 3. Step 1 -Load the Dataset

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/titanic/train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


#### Check the dataset:

In [15]:
df.shape
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# 4. Step 2 - Inspect the Missing Values

In [19]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [18]:
missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

missing_percentage

Cabin          77.104377
Age            19.865320
Embarked        0.224467
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
dtype: float64

##### This tells us which columns require preprocessing.

# 5. Step 3 - Remove Unnecessary Columns

Some columns are identifiers or raw text that we don't need in their original form.

for this project start with:

In [28]:
df = df.drop(
    columns=["PassengerId","Ticket","Cabin","Name"],
    errors="ignore"
)

##### Why?

PassengerId does not represent a meaningful passenger characteristic.

Ticket, Cabin, and Name can contain useful information, but extracting information from them is a more advanced feature-engineering task. For this week's project, we keep the pipeline focused and clean.

# 6. Step 4 — Separate X and y

This is very important.

In [36]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

Now:

X = Features

y = Target

Never accidentally include Survived inside the input features.

# 7. Step 5 — Identify Feature Types

Let's inspect the remaining columns:

In [41]:
X.dtypes

Pclass        int64
Sex          object
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked     object
dtype: object

We can categorize them.

#### Numerical

Age

SibSp

Parch

Fare

#### Categorical

Sex

Embarked

#### Ordinal

Pclass

##### For this project, we can treat Pclass as numerical/ordinal.

# 8. Step 6 — Build the Preprocessing Pipeline

This is the most important part of today's project.

Import:

In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

### Define columns:

In [48]:
numeric_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "Pclass"
]

categorical_features = [
    "Sex",
    "Embarked"
]

# 9. Numerical Pipeline

For numerical features:

### Missing values

Use median.

### Scaling

Use StandardScaler.

In [55]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

### The pipeline performs:

Missing Age
     ->
Median Imputation
     ->
Standard Scaling
     ->
ML-ready numerical feature

# 10.Categorical Pipeline

For categorical features:

### Missing values

Use the most frequent value.

### Encoding

Use One-Hot Encoding.

In [64]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [66]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="OneHotEncoder")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output = False
            )
        )
    ]
)

# 11. Step 7 — Combine Everything

Now use ColumnTransformer:

In [75]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

#### We now have:

                    ┌─ Numerical ── Imputation ── Scaling ─┐
Raw Features ───────┤                                       ├──> Processed Features
                    └─ Categorical ─ Imputation ─ Encoding ┘

# 12. Step 8 — Train/Test Split

Do this before fitting the preprocessing pipeline.

In [84]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

### Why?

We want the test set to remain unseen during preprocessing.

This prevents data leakage.

### Correct:

Training Data
     ->
fit preprocessing
     ->
transform training data

Test Data
     ->
transform using training parameters


### Incorrect:

Entire Dataset
     ->
fit scaler/imputer
     ->
split data

#### That leaks information from the test set.

# 13. Step 9 — Fit and Transform

In [93]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

#### Notice the difference:

### Training

fit_transform()


### Testing

transform()

#### Never:

X_test_processed = preprocessor.fit_transform(X_test)

#### because that would fit preprocessing separately on the test data.

# 14. Step 10 — Inspect the Output
### Check:

In [99]:
X_train_processed.shape

(712, 10)

### and:

In [100]:
X_test_processed.shape

(179, 10)

### Get feature names:

In [104]:
feature_names = preprocessor.get_feature_names_out()

feature_names

array(['numeric__Age', 'numeric__Fare', 'numeric__SibSp',
       'numeric__Parch', 'numeric__Pclass', 'categorical__Sex_female',
       'categorical__Sex_male', 'categorical__Embarked_C',
       'categorical__Embarked_Q', 'categorical__Embarked_S'], dtype=object)

### Convert the processed data into a DataFrame:

In [118]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,numeric__Age,numeric__Fare,numeric__SibSp,numeric__Parch,numeric__Pclass,categorical__Sex_female,categorical__Sex_male,categorical__Embarked_C,categorical__Embarked_Q,categorical__Embarked_S
692,-0.081135,0.513812,-0.465084,-0.466183,0.829568,0.0,1.0,0.0,0.0,1.0
481,-0.081135,-0.662563,-0.465084,-0.466183,-0.370945,0.0,1.0,0.0,0.0,1.0
527,-0.081135,3.955399,-0.465084,-0.466183,-1.571457,0.0,1.0,0.0,0.0,1.0
855,-0.887827,-0.467874,-0.465084,0.727782,0.829568,1.0,0.0,0.0,0.0,1.0
801,0.110934,-0.115977,0.478335,0.727782,-0.370945,1.0,0.0,0.0,0.0,1.0


# 15. Step 11 — Feature Selection

This week also introduced feature selection.

For the mini-project, first understand an important distinction:

## Feature Engineering

Creating or transforming useful representations.

### Examples:

Age → scaled Age

Sex → one-hot encoded Sex
## Feature Selection

Choosing which existing features should remain.

### Examples:

Age

Fare

Sex

Pclass

while removing unnecessary features.

## Optional selection experiment

You can use:

In [124]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

### For example:

In [129]:
selector = SelectKBest(
    score_func=f_classif,
    k="all"
)

For today's main pipeline, do not aggressively remove features just to reduce the column count.

The goal is to understand the workflow first.

# 16. Step 12 — Create Final Processed Dataset

After preprocessing the complete feature matrix appropriately, construct your final dataset.

For example:

In [139]:
X_processed = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out()

processed_df = pd.DataFrame(
    X_processed,
    columns = feature_names
)

processed_df["Survived"] = y.reset_index(drop=True)

### Then:

In [141]:
processed_df.head()

,numeric__Age,numeric__Fare,numeric__SibSp,numeric__Parch,numeric__Pclass,categorical__Sex_female,categorical__Sex_male,categorical__Embarked_C,categorical__Embarked_Q,categorical__Embarked_S,Survived
0,-0.565736,-0.502445,0.432793,-0.473674,0.827377,0.0,1.0,0.0,0.0,1.0,0
1,0.663861,0.786845,0.432793,-0.473674,-1.566107,1.0,0.0,1.0,0.0,0.0,1
2,-0.258337,-0.488854,-0.474545,-0.473674,0.827377,1.0,0.0,0.0,0.0,1.0,1
3,0.433312,0.420730,0.432793,-0.473674,-1.566107,1.0,0.0,0.0,0.0,1.0,1
4,0.433312,-0.486337,-0.474545,-0.473674,0.827377,0.0,1.0,0.0,0.0,1.0,0


### Check:

In [143]:
processed_df.shape

(891, 11)

### And:

In [144]:
processed_df.isnull().sum().sum()

np.int64(0)

### Expected:

0

## No missing values should remain.

# 17. Step 13 — Save the Dataset

In [152]:
processed_df.to_csv(
    "processed_titanic.csv",
    index=False
)

## Verify:

In [153]:
processed_df.to_csv("processed_titanic.csv", index=False)

print("Processed titanic dataset saved successfully.")

Processed titanic dataset saved successfully.
